# 04 — Tree Models: Decision Tree and Random Forest

Train and compare a decision tree and a random forest using the same saved training–validation split and the same 41 input features as logistic regression.

Sections 1–3 provide the shared data setup and evaluation function. Sections 4–5 contain the decision-tree experiment and its recorded findings. Sections 6–7 train and evaluate the random forest.

Both models keep numerical features in their original units and one-hot encode categorical features. Each pipeline fits its own preprocessing on the training partition. Compare training and validation performance, then inspect false alarms and missed attacks.

## 1. Recover the saved split

Load the official training file and recover the records assigned to each partition. Both tree models reuse these records so their results are comparable with each other and with logistic regression.

The assertions check row counts and ensure that no record ID belongs to both partitions.

In [1]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd().parent

# Load the original development data and saved split assignments.
official_train_df = pd.read_csv(project_root / "data" / "raw" / "UNSW_NB15_training-set.csv")
split_assignments = pd.read_csv(project_root / "data" / "processed" / "train_validation_split.csv")

# Get the record IDs assigned to each partition.
fit_ids = split_assignments.loc[
    split_assignments["partition"].eq("training"), "id"
]
validation_ids = split_assignments.loc[
    split_assignments["partition"].eq("validation"), "id"
]

# Recover the corresponding records.
fit_df = official_train_df.loc[
    official_train_df["id"].isin(fit_ids)
].copy()

validation_df = official_train_df.loc[
    official_train_df["id"].isin(validation_ids)
].copy()

# Check that all assigned records were recovered without overlapping IDs.
assert len(fit_df) == len(fit_ids)
assert len(validation_df) == len(validation_ids)
assert set(fit_ids).isdisjoint(validation_ids)
assert len(fit_df) + len(validation_df) == len(official_train_df)

## 2. Define inputs and target

`X` contains traffic measurements; `y` contains the binary answer (`0 = normal`, `1 = attack`).

Exclude `id` because it is a record identifier, and exclude `label` and `attack_cat` because they contain the answers. Also exclude `is_ftp_login`: it duplicates `ct_ftp_cmd` in the development data and has an unresolved discrepancy with its documented binary meaning.

Expected input shapes: **(140272, 41)** for training and **(35069, 41)** for validation. The 41 inputs contain 38 numerical and 3 categorical columns.

In [2]:
# Confirm the duplicate-feature finding in the training partition.
assert fit_df["is_ftp_login"].eq(fit_df["ct_ftp_cmd"]).all()

excluded_columns = ["id", "attack_cat", "label", "is_ftp_login"]

# X contains the traffic measurements supplied to the model.
X_fit = fit_df.drop(columns=excluded_columns)
X_validation = validation_df.drop(columns=excluded_columns)

# y contains the corresponding correct answers.
y_fit = fit_df["label"].copy()
y_validation = validation_df["label"].copy()

print("X_fit:", X_fit.shape)
print("y_fit:", y_fit.shape)
print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)

X_fit: (140272, 41)
y_fit: (140272,)
X_validation: (35069, 41)
y_validation: (35069,)


In [3]:
# These text columns need categorical encoding.
categorical_columns = ["proto", "service", "state"]

# Select the numerical measurements by their stored data types.
numerical_columns = X_fit.select_dtypes(
    include="number"
).columns.tolist()

# Ensure that every input column has been accounted for.
assert set(categorical_columns + numerical_columns) == set(X_fit.columns)

print("Categorical columns:", categorical_columns)
print("Numerical column count:", len(numerical_columns))

Categorical columns: ['proto', 'service', 'state']
Numerical column count: 38


## 3. Import shared tools and define evaluation

A pipeline keeps preprocessing and classification together. Both experiments reuse the evaluation function below. It uses the confusion matrix to calculate accuracy, attack precision, attack recall, and the false-positive rate.

Its results are proportions between 0 and 1. We multiply by 100 only when displaying percentages. The random-forest estimator is imported in its own training section.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix

In [5]:
def summarize_predictions(y_true, y_pred):
    # With labels [0, 1], the four cells are TN, FP, FN, TP.
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    return {
        "accuracy": (tp + tn) / (tn + fp + fn + tp),
        "precision": tp / (tp + fp) if tp + fp else 0.0,
        "recall": tp / (tp + fn) if tp + fn else 0.0,
        "false_positive_rate": fp / (tn + fp) if tn + fp else 0.0,
    }

## 4. Train a decision tree

A tree learns a sequence of feature-based questions. It chooses splits that separate the training labels into less mixed groups. Each final group is a **leaf**, which predicts its majority class.

Trees do not need standard scaling. `passthrough` keeps numerical measurements in their original units; one-hot encoding converts the text categories into indicator columns. `handle_unknown="ignore"` lets the encoder process an unseen category using all zeros for that feature's indicator columns.

Our initial settings are:

- `max_depth=8`: at most eight splits along a path.
- `min_samples_leaf=20`: at least 20 training records per leaf.
- `random_state=42`: make randomized choices reproducible.

These are starting settings, not optimized values. Limiting tree complexity helps reduce the risk of overfitting.

In [6]:
# Trees can use numerical measurements in their original units.
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", "passthrough", numerical_columns),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_columns,
        ),
    ]
)

# Limit tree complexity to reduce the risk of overfitting.
tree_model = Pipeline(
    steps=[
        ("preprocessing", tree_preprocessor),
        (
            "classifier",
            DecisionTreeClassifier(
                max_depth=8,
                min_samples_leaf=20,
                random_state=42,
            ),
        ),
    ]
)

# Learn preprocessing and tree rules using training records only.
tree_model.fit(X_fit, y_fit)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](41,)","['dur','proto','service',...,'ct_src_ltm','ct_srv_dst','is_sm_ips_ports']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,41
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='

## 5. Evaluate the decision tree

Use the same fitted tree to predict both partitions. `predict()` applies the learned preprocessing and rules without retraining.

Compare the two rows for a training–validation gap. In the confusion matrix, rows are actual labels and columns are predicted labels. A normal record predicted as an attack is a **false positive**; an attack predicted as normal is a **false negative**.

The following cells also record F1 at the default decision threshold and summarize the validation probability scores with ROC AUC and average precision.

In [7]:
# Evaluate the same fitted tree on both partitions.
tree_training_predictions = tree_model.predict(X_fit)
tree_predictions = tree_model.predict(X_validation)

tree_partition_results = pd.DataFrame(
    {
        "Training": summarize_predictions(
            y_fit, tree_training_predictions
        ),
        "Validation": summarize_predictions(
            y_validation, tree_predictions
        ),
    }
).T

# Show metric values as percentages.
display((tree_partition_results * 100).round(2))

# Count correct predictions, false alarms, and missed attacks.
tree_confusion = pd.DataFrame(
    confusion_matrix(y_validation, tree_predictions, labels=[0, 1]),
    index=["Actual normal", "Actual attack"],
    columns=["Predicted normal", "Predicted attack"],
)

display(tree_confusion)

,accuracy,precision,recall,false_positive_rate
Training,94.18,94.68,96.89,11.60
Validation,94.39,94.77,97.11,11.42


,Predicted normal,Predicted attack
Actual normal,9921,1279
Actual attack,689,23180


In [8]:
from sklearn.metrics import roc_auc_score, average_precision_score

# Find the probability column for attack (label 1).
tree_attack_column = list(tree_model.classes_).index(1)

# Get validation probability scores from the fitted pipeline.
tree_attack_probabilities = tree_model.predict_proba(
    X_validation
)[:, tree_attack_column]

# Calculate F1 from the default predictions.
tn, fp, fn, tp = confusion_matrix(
    y_validation, tree_predictions, labels=[0, 1]
).ravel()

tree_f1 = 2 * tp / (2 * tp + fp + fn)

# Evaluate ranking across thresholds using probability scores.
tree_roc_auc = roc_auc_score(
    y_validation, tree_attack_probabilities
)
tree_ap = average_precision_score(
    y_validation, tree_attack_probabilities
)

print(f"Decision-tree validation F1:      {tree_f1:.2%}")
print(f"Decision-tree validation ROC AUC: {tree_roc_auc:.4f}")
print(f"Decision-tree validation AP:      {tree_ap:.4f}")

Decision-tree validation F1:      95.93%
Decision-tree validation ROC AUC: 0.9872
Decision-tree validation AP:      0.9928


### Decision-tree findings

The initial tree uses max_depth=8 and min_samples_leaf=20.

- Training accuracy: 94.18%.
- Validation accuracy: 94.39%.
- Validation attack precision: 94.77%.
- Validation attack recall: 97.11%.
- Validation false-positive rate: 11.42%.
- Validation attack F1: 95.93%.
- Validation ROC AUC: 0.9872.
- Validation average precision: 0.9928.

Training and validation performance are similar, with no large gap.

Compared with logistic regression at the default threshold, the tree
produces 650 fewer false alarms but misses 403 additional attacks.
Its slightly higher summary scores do not resolve this tradeoff.

The official test set remains unused for model evaluation.

## 6. Train a random forest

A random forest combines many decision trees. Each tree trains on a random sample of training records drawn **with replacement**, so some records repeat and others are omitted. At each split, it considers a random subset of features. The forest averages its trees' predicted probabilities.

These differences between trees help reduce sensitivity to a particular training sample, but a forest is not guaranteed to outperform a single tree.

We reuse the shared inputs and evaluation function from sections 1–3. Numerical values remain unchanged, and categorical values are one-hot encoded in a separate pipeline.

Initial settings:

- `n_estimators=100`: build 100 trees.
- `max_depth=8` and `min_samples_leaf=20`: retain the initial complexity limits used for the single tree.
- `max_features="sqrt"`: consider approximately the square root of the prepared feature count when looking for a split.
- `bootstrap=True`: sample training records with replacement for each tree.
- `random_state=42`: make randomized choices reproducible.
- `n_jobs=-1`: use all available CPU cores for parallel tree building.

These are starting settings, not optimized values. Fit on the training partition only.

In [9]:
from sklearn.ensemble import RandomForestClassifier

# Apply the same preprocessing choices as the single-tree experiment.
forest_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", "passthrough", numerical_columns),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_columns,
        ),
    ]
)

# Combine 100 trees, each with limits on its complexity.
forest_model = Pipeline(
    steps=[
        ("preprocessing", forest_preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=100,
                max_depth=8,
                min_samples_leaf=20,
                max_features="sqrt",
                bootstrap=True,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

# Fit preprocessing and all trees using training records only.
forest_model.fit(X_fit, y_fit)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](41,)","['dur','proto','service',...,'ct_src_ltm','ct_srv_dst','is_sm_ips_ports']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,41
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='

## 7. Evaluate the random forest

Predict both partitions with the fitted forest. Reuse the same metric definitions and label order as the decision-tree experiment.

Compare training and validation performance for a gap. Then compare the validation results with the decision-tree results above, considering false alarms and missed attacks as well as accuracy.

Run the new cells to produce results; no random-forest findings have been recorded yet.

In [10]:
# Predict with the same fitted forest on both partitions.
forest_training_predictions = forest_model.predict(X_fit)
forest_predictions = forest_model.predict(X_validation)

forest_partition_results = pd.DataFrame(
    {
        "Training": summarize_predictions(
            y_fit, forest_training_predictions
        ),
        "Validation": summarize_predictions(
            y_validation, forest_predictions
        ),
    }
).T

# Display percentages to inspect the training–validation gap.
display((forest_partition_results * 100).round(2))

# Count correct predictions, false alarms, and missed attacks.
forest_confusion = pd.DataFrame(
    confusion_matrix(y_validation, forest_predictions, labels=[0, 1]),
    index=["Actual normal", "Actual attack"],
    columns=["Predicted normal", "Predicted attack"],
)

display(forest_confusion)

,accuracy,precision,recall,false_positive_rate
Training,93.42,91.19,99.99,20.58
Validation,93.59,91.41,99.98,20.03


,Predicted normal,Predicted attack
Actual normal,8957,2243
Actual attack,4,23865


In [11]:
# Find the probability column for attack (label 1).
forest_attack_column = list(forest_model.classes_).index(1)

# Get validation probability scores from the fitted forest.
forest_attack_probabilities = forest_model.predict_proba(
    X_validation
)[:, forest_attack_column]

# Calculate F1 from the default predictions.
forest_tn, forest_fp, forest_fn, forest_tp = confusion_matrix(
    y_validation, forest_predictions, labels=[0, 1]
).ravel()

forest_f1 = (
    2 * forest_tp
    / (2 * forest_tp + forest_fp + forest_fn)
)

# These functions were imported in the decision-tree evaluation section.
forest_roc_auc = roc_auc_score(
    y_validation, forest_attack_probabilities
)
forest_ap = average_precision_score(
    y_validation, forest_attack_probabilities
)

print(f"Random-forest validation F1:      {forest_f1:.2%}")
print(f"Random-forest validation ROC AUC: {forest_roc_auc:.4f}")
print(f"Random-forest validation AP:      {forest_ap:.4f}")

Random-forest validation F1:      95.50%
Random-forest validation ROC AUC: 0.9880
Random-forest validation AP:      0.9942


In [12]:
forest_threshold_rows = []

for threshold in [0.5, 0.6, 0.7, 0.8, 0.9]:
    # Raise an alarm only when the attack probability exceeds the cutoff.
    predictions_at_threshold = (
        forest_attack_probabilities > threshold
    ).astype(int)

    # Calculate the same metrics for every threshold.
    metrics = summarize_predictions(
        y_validation, predictions_at_threshold
    )

    # Include counts so the practical tradeoff is visible.
    tn, fp, fn, tp = confusion_matrix(
        y_validation, predictions_at_threshold, labels=[0, 1]
    ).ravel()

    metrics["false_alarms"] = fp
    metrics["missed_attacks"] = fn
    metrics["threshold"] = threshold
    forest_threshold_rows.append(metrics)

forest_threshold_results = pd.DataFrame(
    forest_threshold_rows
).set_index("threshold")

# Convert only rate columns to percentages; keep counts unchanged.
forest_threshold_display = forest_threshold_results.copy()
rate_columns = ["accuracy", "precision", "recall", "false_positive_rate"]
forest_threshold_display[rate_columns] *= 100

display(forest_threshold_display.round(2))

,accuracy,precision,recall,false_positive_rate,false_alarms,missed_attacks
threshold,,,,,,
0.5,93.59,91.41,99.98,20.03,2243,4
0.6,94.24,92.92,99.09,16.10,1803,217
0.7,93.95,96.34,94.72,7.67,859,1261
0.8,91.13,98.09,88.70,3.69,413,2698
0.9,83.68,99.75,76.21,0.41,46,5679


In [13]:
import xgboost

# Confirm that the notebook can access the installed package.
print("XGBoost version:", xgboost.__version__)

XGBoost version: 3.4.1


In [14]:
from sklearn.base import clone
from xgboost import XGBClassifier

# Reuse the preprocessing configuration in a fresh, unfitted copy.
xgb_preprocessor = clone(tree_preprocessor)

# Keep zeros explicitly stored, because XGBoost treats omitted
# entries in a sparse matrix as missing.
xgb_preprocessor.set_params(sparse_threshold=0)

xgb_model = Pipeline(
    steps=[
        ("preprocessing", xgb_preprocessor),
        (
            "classifier",
            XGBClassifier(
                n_estimators=200,
                max_depth=4,
                learning_rate=0.1,
                objective="binary:logistic",
                tree_method="hist",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

# Learn preprocessing and boosted trees from training records only.
xgb_model.fit(X_fit, y_fit)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](41,)","['dur','proto','service',...,'ct_src_ltm','ct_srv_dst','is_sm_ips_ports']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,41
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When 

In [15]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)


def evaluate_model(model):
    # Check training performance and evaluate unseen validation records.
    training_predictions = model.predict(X_fit)
    validation_predictions = model.predict(X_validation)

    # Extract attack probabilities for ranking metrics.
    attack_column = list(model.classes_).index(1)
    attack_probabilities = model.predict_proba(
        X_validation
    )[:, attack_column]

    # Reuse our existing precision, recall, accuracy, and FPR calculations.
    metrics = summarize_predictions(y_validation, validation_predictions)
    metrics["validation_accuracy"] = metrics.pop("accuracy")
    metrics["training_accuracy"] = accuracy_score(
        y_fit, training_predictions
    )
    metrics["f1"] = f1_score(y_validation, validation_predictions)
    metrics["roc_auc"] = roc_auc_score(
        y_validation, attack_probabilities
    )
    metrics["average_precision"] = average_precision_score(
        y_validation, attack_probabilities
    )

    return metrics


# Compare all three fitted tree models using the same evaluation.
model_comparison = pd.DataFrame(
    {
        "Decision tree": evaluate_model(tree_model),
        "Random forest": evaluate_model(forest_model),
        "XGBoost": evaluate_model(xgb_model),
    }
).T

# Display familiar rates as percentages; keep AUC and AP between 0 and 1.
percentage_columns = [
    "training_accuracy",
    "validation_accuracy",
    "precision",
    "recall",
    "false_positive_rate",
    "f1",
]

comparison_display = model_comparison[
    percentage_columns + ["roc_auc", "average_precision"]
].copy()

comparison_display[percentage_columns] = (
    comparison_display[percentage_columns] * 100
).round(2)

display(comparison_display.round(4))

,training_accuracy,validation_accuracy,precision,recall,false_positive_rate,f1,roc_auc,average_precision
Decision tree,94.18,94.39,94.77,97.11,11.42,95.93,0.9872,0.9928
Random forest,93.42,93.59,91.41,99.98,20.03,95.50,0.9880,0.9942
XGBoost,95.44,95.20,95.62,97.40,9.51,96.50,0.9916,0.9960


In [16]:
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold

# Keep identical input patterns together within cross-validation.
training_groups = X_fit.groupby(
    list(X_fit.columns),
    dropna=False,
    sort=False,
).ngroup()

# Rotate through three folds within the training partition.
grouped_cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42,
)

# Try four combinations, including our original depth=4, trees=200.
parameter_grid = {
    "classifier__max_depth": [4, 6],
    "classifier__n_estimators": [100, 200],
}

xgb_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=parameter_grid,
    scoring="f1",
    cv=grouped_cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score="raise",
)

# Each fold fits its own preprocessing and model.
# Neither our separate validation partition nor the official test is used.
xgb_search.fit(X_fit, y_fit, groups=training_groups)

print("Best settings:", xgb_search.best_params_)
print(f"Best mean cross-validation F1: {xgb_search.best_score_:.4f}")

# Show all four candidates, including variation across folds.
search_results = pd.DataFrame(xgb_search.cv_results_)

display(
    search_results[
        [
            "param_classifier__max_depth",
            "param_classifier__n_estimators",
            "mean_test_score",
            "std_test_score",
            "rank_test_score",
        ]
    ].sort_values("rank_test_score").round(4)
)

Fitting 3 folds for each of 4 candidates, totalling 12 fits
Best settings: {'classifier__max_depth': 6, 'classifier__n_estimators': 200}
Best mean cross-validation F1: 0.9665


,param_classifier__max_depth,param_classifier__n_estimators,mean_test_score,std_test_score,rank_test_score
3,6,200,0.9665,0.0007,1
1,4,200,0.9651,0.0008,2
2,6,100,0.9648,0.0007,3
0,4,100,0.9634,0.0010,4


In [17]:
# GridSearchCV already refitted the winning pipeline on all training records.
tuned_xgb_model = xgb_search.best_estimator_

# Evaluate it using our existing function and record the result.
model_comparison.loc["XGBoost (tuned)"] = evaluate_model(tuned_xgb_model)

# Compare the original and tuned versions.
tuning_comparison = model_comparison.loc[
    ["XGBoost", "XGBoost (tuned)"],
    percentage_columns + ["roc_auc", "average_precision"],
].copy()

# Display rates as percentages, keeping AUC and AP between 0 and 1.
tuning_comparison[percentage_columns] = (
    tuning_comparison[percentage_columns] * 100
).round(2)

display(tuning_comparison.round(4))

,training_accuracy,validation_accuracy,precision,recall,false_positive_rate,f1,roc_auc,average_precision
XGBoost,95.44,95.20,95.62,97.40,9.51,96.50,0.9916,0.9960
XGBoost (tuned),96.33,95.66,96.11,97.58,8.42,96.84,0.9930,0.9966


In [18]:
# Get attack probabilities from the tuned model.
attack_column = list(tuned_xgb_model.classes_).index(1)
tuned_attack_probabilities = tuned_xgb_model.predict_proba(
    X_validation
)[:, attack_column]

threshold_rows = []

for threshold in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    # Apply the cutoff without retraining the model.
    predictions = (tuned_attack_probabilities > threshold).astype(int)

    metrics = summarize_predictions(y_validation, predictions)
    metrics["f1"] = f1_score(y_validation, predictions)

    # Include error counts so we can understand the tradeoff.
    tn, fp, fn, tp = confusion_matrix(
        y_validation, predictions, labels=[0, 1]
    ).ravel()

    metrics["false_alarms"] = fp
    metrics["missed_attacks"] = fn
    metrics["threshold"] = threshold
    threshold_rows.append(metrics)

tuned_threshold_results = pd.DataFrame(
    threshold_rows
).set_index("threshold")

# Select by full-precision F1, then recall as the tie-breaker.
selected_threshold = tuned_threshold_results.sort_values(
    ["f1", "recall"],
    ascending=[False, False],
).index[0]

# Format rates as percentages while preserving error counts.
rate_columns = ["precision", "recall", "false_positive_rate", "f1"]
threshold_display = tuned_threshold_results[
    rate_columns + ["false_alarms", "missed_attacks"]
].copy()

threshold_display[rate_columns] *= 100

display(threshold_display.round(2))
print(f"Selected validation threshold: {selected_threshold:.1f}")

,precision,recall,false_positive_rate,f1,false_alarms,missed_attacks
threshold,,,,,,
0.3,93.04,99.59,15.88,96.20,1779,97
0.4,94.50,98.84,12.26,96.62,1373,277
0.5,96.11,97.58,8.42,96.84,943,578
0.6,97.26,95.89,5.77,96.57,646,981
0.7,98.09,93.81,3.90,95.90,437,1478
0.8,98.81,91.16,2.35,94.83,263,2109
0.9,99.55,87.28,0.84,93.01,94,3037


Selected validation threshold: 0.5


### Selected model and threshold

Selected model: XGBoost with 200 trees, maximum depth 6,
and learning rate 0.1.

Settings were selected using three-fold grouped cross-validation
within the training partition, optimizing attack F1.

The decision threshold is 0.5, selected by validation attack F1
from seven candidate thresholds.

The exact fitted pipeline and threshold are now fixed for evaluation
on the reserved official test set.

In [19]:
# Freeze the fitted pipeline and selected threshold.
final_model = tuned_xgb_model
final_threshold = 0.5

# Load the reserved official test set for final evaluation.
official_test_df = pd.read_csv(
    project_root / "data" / "raw" / "UNSW_NB15_testing-set.csv"
)

# Use exactly the same input columns and ordering as training.
X_test = official_test_df[X_fit.columns].copy()
y_test = official_test_df["label"].copy()

# Apply the already-fitted preprocessing and model.
attack_column = list(final_model.classes_).index(1)
test_attack_probabilities = final_model.predict_proba(
    X_test
)[:, attack_column]

test_predictions = (
    test_attack_probabilities > final_threshold
).astype(int)

# Calculate all final metrics together.
test_metrics = summarize_predictions(y_test, test_predictions)
test_metrics["f1"] = f1_score(y_test, test_predictions)
test_metrics["roc_auc"] = roc_auc_score(
    y_test, test_attack_probabilities
)
test_metrics["average_precision"] = average_precision_score(
    y_test, test_attack_probabilities
)

# Display classification rates as percentages.
test_display = pd.DataFrame([test_metrics], index=["Official test"])
test_rate_columns = [
    "accuracy", "precision", "recall", "false_positive_rate", "f1"
]
test_display[test_rate_columns] = (
    test_display[test_rate_columns] * 100
).round(2)

display(test_display.round(4))

# Show the final error counts.
test_confusion = pd.DataFrame(
    confusion_matrix(y_test, test_predictions, labels=[0, 1]),
    index=["Actual normal", "Actual attack"],
    columns=["Predicted normal", "Predicted attack"],
)

display(test_confusion)

,accuracy,precision,recall,false_positive_rate,f1,roc_auc,average_precision
Official test,87.57,82.51,98.27,25.53,89.7,0.9838,0.9882


,Predicted normal,Predicted attack
Actual normal,27555,9445
Actual attack,785,44547


In [20]:
import json

# Save final results in a reusable format for the report and README.
metrics_directory = project_root / "reports" / "metrics"
metrics_directory.mkdir(parents=True, exist_ok=True)

classifier_parameters = final_model.named_steps["classifier"].get_params()

final_report = {
    "model": "XGBoost",
    "model_parameters": {
        name: classifier_parameters[name]
        for name in ["n_estimators", "max_depth", "learning_rate"]
    },
    "threshold": float(final_threshold),
    "evaluation_dataset": "UNSW_NB15_testing-set.csv",
    "training_rows": len(X_fit),
    "validation_rows": len(X_validation),
    "test_rows": len(X_test),
    "metrics": {
        name: float(value)
        for name, value in test_metrics.items()
    },
    "confusion_matrix_label_order": [0, 1],
    "confusion_matrix_rows": "actual labels",
    "confusion_matrix_columns": "predicted labels",
    "confusion_matrix": confusion_matrix(
        y_test, test_predictions, labels=[0, 1]
    ).tolist(),
}

report_path = metrics_directory / "final_test_metrics.json"

with report_path.open("w", encoding="utf-8") as file:
    json.dump(final_report, file, indent=2)

print("Saved:", report_path)

Saved: C:\Code\fourthyear\resume_projects\network_intrusion_detection_system\reports\metrics\final_test_metrics.json


In [21]:
# Attach final predictions to descriptive columns for analysis.
test_errors = official_test_df[
    ["label", "service", "attack_cat"]
].copy()

test_errors["prediction"] = test_predictions
test_errors["incorrect"] = (
    test_errors["label"] != test_errors["prediction"]
)

# Among actual normal records, an incorrect prediction is a false alarm.
normal_errors_by_service = (
    test_errors.loc[test_errors["label"].eq(0)]
    .groupby("service")
    .agg(
        normal_records=("label", "size"),
        false_alarms=("incorrect", "sum"),
    )
)

normal_errors_by_service["false_alarm_percentage"] = (
    normal_errors_by_service["false_alarms"]
    / normal_errors_by_service["normal_records"]
    * 100
)

print("Normal traffic: services contributing the most false alarms")
display(
    normal_errors_by_service
    .sort_values("false_alarms", ascending=False)
    .round(2)
)

# Among actual attacks, an incorrect prediction is a missed attack.
misses_by_category = (
    test_errors.loc[test_errors["label"].eq(1)]
    .groupby("attack_cat")
    .agg(
        attack_records=("label", "size"),
        missed_attacks=("incorrect", "sum"),
    )
)

misses_by_category["miss_percentage"] = (
    misses_by_category["missed_attacks"]
    / misses_by_category["attack_records"]
    * 100
)

print("Attack categories: highest missed-attack rates")
display(
    misses_by_category
    .sort_values("miss_percentage", ascending=False)
    .round(2)
)

Normal traffic: services contributing the most false alarms


,normal_records,false_alarms,false_alarm_percentage
service,,,
-,27375,7968,29.11
http,4013,1164,29.01
ftp,758,310,40.90
radius,2,2,100.00
dns,3068,1,0.03
ftp-data,949,0,0.00
smtp,635,0,0.00
ssh,200,0,0.00


Attack categories: highest missed-attack rates


,attack_records,missed_attacks,miss_percentage
attack_cat,,,
Fuzzers,6062,693,11.43
Analysis,677,20,2.95
Shellcode,378,3,0.79
Exploits,11132,61,0.55
Reconnaissance,3496,4,0.11
DoS,4089,3,0.07
Generic,18871,1,0.01
Backdoor,583,0,0.00
Worms,44,0,0.00


In [22]:
# Preserve the error-analysis tables alongside the final metrics.
metrics_directory = project_root / "reports" / "metrics"
metrics_directory.mkdir(parents=True, exist_ok=True)

normal_errors_by_service.to_csv(
    metrics_directory / "test_false_alarms_by_service.csv"
)
misses_by_category.to_csv(
    metrics_directory / "test_misses_by_attack_category.csv"
)

In [23]:
# Access the fitted components of our selected pipeline.
fitted_preprocessor = final_model.named_steps["preprocessing"]
fitted_classifier = final_model.named_steps["classifier"]

# Match each prepared feature name with its importance score.
feature_importance = pd.DataFrame(
    {
        "feature": fitted_preprocessor.get_feature_names_out(),
        "relative_gain": fitted_classifier.feature_importances_,
    }
).sort_values(
    "relative_gain", ascending=False
).reset_index(drop=True)

# Inspect the 15 highest-ranked prepared features.
display(feature_importance.head(15).round(4))

# Save the complete ranking for the report.
metrics_directory = project_root / "reports" / "metrics"
metrics_directory.mkdir(parents=True, exist_ok=True)

feature_importance.to_csv(
    metrics_directory / "feature_importance_gain.csv",
    index=False,
)

,feature,relative_gain
0,numeric__sttl,0.7109
1,numeric__ct_srv_dst,0.0333
2,numeric__sbytes,0.0187
3,numeric__swin,0.0151
4,categorical__service_dns,0.0150
5,numeric__ct_dst_sport_ltm,0.0147
6,numeric__smean,0.0131
7,numeric__sloss,0.0100
8,categorical__state_CON,0.0096
9,categorical__service_http,0.0096


### Feature-importance finding

Source-to-destination TTL (`sttl`) dominates the normalized average-gain
ranking at 0.7109. Other leading features include `ct_srv_dst`, `sbytes`,
`swin`, and the DNS service indicator.

Gain importance describes training split improvements. It does not
establish causation, indicate the direction of an effect, or explain
an individual prediction.

In [24]:
import joblib
import platform
from importlib.metadata import version

models_directory = project_root / "models"
models_directory.mkdir(parents=True, exist_ok=True)

# Bundle the fitted pipeline with the information needed for prediction.
model_bundle = {
    "pipeline": final_model,
    "threshold": float(final_threshold),
    "positive_label": 1,
    "feature_columns": X_fit.columns.tolist(),
    "categorical_columns": categorical_columns,
    "numerical_columns": numerical_columns,
    "versions": {
        "python": platform.python_version(),
        **{
            package: version(package)
            for package in [
                "scikit-learn", "xgboost", "pandas",
                "numpy", "scipy", "joblib",
            ]
        },
    },
}

model_path = models_directory / "intrusion_detector.joblib"

# Save learned settings and metadata without retraining.
joblib.dump(model_bundle, model_path, compress=3)

print("Saved:", model_path)
print(f"File size: {model_path.stat().st_size / 1024**2:.2f} MB")

Saved: C:\Code\fourthyear\resume_projects\network_intrusion_detection_system\models\intrusion_detector.joblib
File size: 0.17 MB


In [26]:
import numpy as np

# Reload our own saved artifact and compare a few predictions.
loaded_bundle = joblib.load(model_path)
sample = X_validation.head(10)

np.testing.assert_allclose(
    final_model.predict_proba(sample),
    loaded_bundle["pipeline"].predict_proba(sample),
)

print("Save/load check passed: probabilities match.")

Save/load check passed: probabilities match.


In [27]:
# Save one validation record as a JSON object for the standalone script.
example_path = project_root / "data" / "processed" / "example_record.json"
X_validation.iloc[0].to_json(example_path, indent=2)

print("Saved example:", example_path)

Saved example: C:\Code\fourthyear\resume_projects\network_intrusion_detection_system\data\processed\example_record.json


In [28]:
# Load both independently saved artifacts.
original_bundle = joblib.load(
    project_root / "models" / "intrusion_detector.joblib"
)
scripted_bundle = joblib.load(
    project_root / "models" / "reproduced_intrusion_detector.joblib"
)

# Check that both artifacts expect the same inputs and decision threshold.
assert original_bundle["feature_columns"] == scripted_bundle["feature_columns"]
assert original_bundle["threshold"] == scripted_bundle["threshold"]


def get_attack_probabilities(bundle):
    pipeline = bundle["pipeline"]
    attack_column = list(pipeline.classes_).index(bundle["positive_label"])
    return pipeline.predict_proba(X_validation)[:, attack_column]


original_probabilities = get_attack_probabilities(original_bundle)
scripted_probabilities = get_attack_probabilities(scripted_bundle)

# Allow tiny floating-point differences in probability calculations.
np.testing.assert_allclose(
    original_probabilities,
    scripted_probabilities,
    rtol=1e-6,
    atol=1e-8,
)

# The final predicted labels must match exactly.
np.testing.assert_array_equal(
    original_probabilities > original_bundle["threshold"],
    scripted_probabilities > scripted_bundle["threshold"],
)

print(
    f"Reproduction check passed on {len(X_validation):,} validation records."
)

Reproduction check passed on 35,069 validation records.
